# AI4MH Governance Layer — Walkthrough
### GSoC 2026 | ISSR, University of Alabama

This notebook walks through the complete AI4MH governance pipeline end-to-end.
It demonstrates how the system evaluates county-level social media signals,
applies governance flags, computes a Crisis Signal Score (CSS), and routes
signals to human reviewers with full uncertainty transparency.

**Three demo scenarios are shown:**
- Jefferson County — genuine crisis signal
- Madison County — media-driven spike (noise)
- Lowndes County — rural sparse data (routed to human review queue)

---
> **North Star:** A trend intelligence tool for public health decision-makers.
> No score autonomously triggers action. Human confirmation is always required.

## Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from datetime import datetime, timedelta
import json

import config
from ai4mh.flags import Post, run_all_flags
from ai4mh.scoring import evaluate_county, EscalationTier
from ai4mh.audit import AuditLogger, AnalystAction
from ai4mh.pipeline import _generate_sample_data, COUNTY_METADATA

print('✅ All modules loaded successfully')
print(f'Analysis window: {config.ANALYSIS_WINDOW_HOURS} hours')
print(f'Urban min posts: {config.MIN_POSTS_URBAN} | Rural min posts: {config.MIN_POSTS_RURAL}')
print(f'CSS thresholds — HIGH: {config.CSS_HIGH_THRESHOLD} | MODERATE: {config.CSS_MODERATE_THRESHOLD} | LOW: {config.CSS_LOW_THRESHOLD}')

✅ All modules loaded successfully
Analysis window: 72 hours
Urban min posts: 30 | Rural min posts: 15
CSS thresholds — HIGH: 0.75 | MODERATE: 0.5 | LOW: 0.3


## Step 1 — Load Sample Data

Three Alabama counties are evaluated over a 72-hour window.
Each county represents a different signal scenario.

In [12]:
posts_by_county = _generate_sample_data()
window_start = datetime.utcnow() - timedelta(hours=config.ANALYSIS_WINDOW_HOURS)

print(f'Analysis window start: {window_start.strftime("%Y-%m-%d %H:%M")} UTC\n')
print(f'{"County":<25} {"FIPS":<8} {"Posts":<8} {"RUCC":<6} {"Type"}')
print('-' * 60)

for fips, posts in posts_by_county.items():
    meta = COUNTY_METADATA.get(fips, {})
    county_type = 'Rural' if meta.get('rucc_code', 1) >= 7 else 'Urban'
    print(f"{meta.get('name','?'):<25} {fips:<8} {len(posts):<8} {meta.get('rucc_code','?'):<6} {county_type}")

Analysis window start: 2026-03-01 18:03 UTC

County                    FIPS     Posts    RUCC   Type
------------------------------------------------------------
Jefferson County          01073    40       1      Urban
Madison County            01089    40       2      Urban
Lowndes County            01085    8        8      Rural


C:\Users\jains\AppData\Local\Temp\ipykernel_12360\3441219323.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  window_start = datetime.utcnow() - timedelta(hours=config.ANALYSIS_WINDOW_HOURS)


## Step 2 — Sample Posts Preview

A preview of the first few posts from each county.

In [13]:
for fips, posts in posts_by_county.items():
    meta = COUNTY_METADATA.get(fips, {})
    print(f"\n{'='*60}")
    print(f"📍 {meta.get('name')} ({fips}) — first 3 posts")
    print('='*60)
    for p in posts[:3]:
        print(f"  [{p.account_id}] {p.text[:80]}..." if len(p.text) > 80 else f"  [{p.account_id}] {p.text}")


📍 Jefferson County (01073) — first 3 posts
  [user_0000] I can't do this anymore. Everything feels hopeless and I don't see a way out.
  [user_0001] Been having really dark thoughts lately. Can't sleep, can't eat. I'm exhausted.
  [user_0002] I don't want to be here anymore. The pain is too much.

📍 Madison County (01089) — first 3 posts
  [user_0000] Did you hear about the suicide case reported in the news today? So sad.
  [user_0001] Breaking news: mental health crisis reported locally. Have you seen this?
  [user_0002] According to the article, there's been an increase in mental health issues.

📍 Lowndes County (01085) — first 3 posts
  [user_0000] I'm not doing well. Really struggling out here and there's nowhere to go.
  [user_0001] Feeling real low. Nothing seems worth it anymore.
  [user_0002] Been a hard week. Can't shake this feeling of hopelessness.


## Step 3 — Governance Flag Detection

Before scoring, the system checks for conditions that modify signal reliability:
- **bot_risk** — velocity filter + near-duplicate clustering
- **media_context** — temporal alignment with news events + linguistic register
- **rural_sparse** — below minimum sample threshold
- **rural_bias** — model trained on urban data, may underdetect rural distress

Flags **never suppress** a signal. They reduce volume weight and surface the reason to the human reviewer.

In [14]:
flag_results = {}

for fips, posts in posts_by_county.items():
    meta = COUNTY_METADATA.get(fips, {})
    flag_result = run_all_flags(
        posts=posts,
        county_fips=fips,
        rucc_code=meta['rucc_code'],
        window_start=window_start
    )
    flag_results[fips] = flag_result

    print(f"\n{'='*55}")
    print(f"🚩 {meta.get('name')} ({fips})")
    print('='*55)

    if flag_result.active_flags:
        print(f"  Active flags: {', '.join(flag_result.active_flags)}")
        print(f"  Volume discount: {flag_result.volume_discount:.0%}")
        print(f"  Discounted posts: {len(flag_result.discounted_posts)}")
        print(f"\n  Reviewer notes:")
        for note in flag_result.plain_language_notes:
            print(f"  • {note}")
    else:
        print('  No flags detected — signal unmodified')


🚩 Jefferson County (01073)
  Active flags: bot_risk
  Volume discount: 100%
  Discounted posts: 2

  Reviewer notes:
  • Near-duplicate clustering: 2 posts identified as coordinated (high text similarity within 6 hours). Cluster contributes 1 post-weight to volume.

🚩 Madison County (01089)
  Active flags: media_context, bot_risk
  Volume discount: 60%
  Discounted posts: 6

  Reviewer notes:
  • Near-duplicate clustering: 6 posts identified as coordinated (high text similarity within 6 hours). Cluster contributes 1 post-weight to volume.
  • 100% of posts classified as reportative (discussing news rather than personal distress). Volume component discounted to reflect genuine distress signal.

🚩 Lowndes County (01085)
  Active flags: rural_sparse, rural_bias
  Volume discount: 100%
  Discounted posts: 0

  Reviewer notes:
  • Sample size below threshold (8 posts, minimum 15 for rural county). Signal routed to sparse-data review queue. Treat confidence estimate with additional caution.

## Step 4 — Conditional CSS Scoring

The Crisis Signal Score (CSS) uses conditional weighting:

- If **sentiment is extreme** (> 0.85) → sentiment alone is sufficient, no corroboration needed
- If **sentiment is moderate** → volume and geography must corroborate
- Volume is **discounted** when bot or media flags are active

This reflects reality: a volume spike from bots tells you nothing about genuine distress.

In [15]:
css_results = {}

for fips, posts in posts_by_county.items():
    meta = COUNTY_METADATA.get(fips, {})
    result = evaluate_county(
        posts=posts,
        county_fips=fips,
        rucc_code=meta['rucc_code'],
        window_start=window_start,
        baseline_mean=meta['baseline_mean'],
        baseline_std=meta['baseline_std'],
        flag_result=flag_results[fips]
    )
    css_results[fips] = result

    tier_icons = {'HIGH': '🔴', 'MODERATE': '🟡', 'LOW': '🟢', 'NOISE': '⚪'}
    icon = tier_icons.get(result.escalation_tier.value, '?')

    print(f"\n{'='*55}")
    print(f"{icon} {meta.get('name')} ({fips})")
    print('='*55)

    if result.routed_to_sparse_queue:
        print('  ⚠  ROUTED TO SPARSE-DATA REVIEW QUEUE')
        print(f'  Posts: {result.n_posts} (below threshold)')
    else:
        print(f"  Posts: {result.n_posts}")
        print(f"  Component scores:")
        print(f"    Sentiment (anchor):     {result.components.sentiment:.3f}")
        print(f"    Volume (raw):           {result.components.volume:.3f}")
        print(f"    Volume (adjusted):      {result.components.volume_adjusted:.3f}")
        print(f"    Geography:              {result.components.geography:.3f}")
        print(f"  ──────────────────────────────")
        print(f"  CSS Score:              {result.css:.3f}")
        print(f"  Escalation Tier:        {result.escalation_tier.value}")

    print(f"\n  Confidence ({result.confidence.percentage:.0%} | {result.confidence.visual_tier}):")
    print(f"  {result.confidence.plain_language}")


🟢 Jefferson County (01073)
  Posts: 40
  Component scores:
    Sentiment (anchor):     0.284
    Volume (raw):           0.646
    Volume (adjusted):      0.646
    Geography:              0.000
  ──────────────────────────────
  CSS Score:              0.354
  Escalation Tier:        LOW

  Confidence (10% | GRAY):
  Signal driven by significant volume spike — Caution: flags active: bot_risk.

⚪ Madison County (01089)
  Posts: 40
  Component scores:
    Sentiment (anchor):     0.007
    Volume (raw):           0.889
    Volume (adjusted):      0.533
    Geography:              0.000
  ──────────────────────────────
  CSS Score:              0.190
  Escalation Tier:        NOISE

  Confidence (10% | GRAY):
  Signal driven by weak signal across components — Caution: flags active: media_context, bot_risk.

🟢 Lowndes County (01085)
  ⚠  ROUTED TO SPARSE-DATA REVIEW QUEUE
  Posts: 8 (below threshold)

  Confidence (20% | GRAY):
  Insufficient data (8 posts, minimum 15 for rural county). R

## Step 5 — Escalation Summary

All counties are classified and routed. No action is taken automatically.
The human reviewer sees severity, evidence, flags, and confidence — all together.

In [16]:
print('ESCALATION SUMMARY')
print('='*65)
print(f'{"County":<25} {"CSS":<8} {"Tier":<12} {"Confidence":<12} {"Flags"}')
print('-'*65)

tier_icons = {'HIGH': '🔴', 'MODERATE': '🟡', 'LOW': '🟢', 'NOISE': '⚪'}

for fips, result in css_results.items():
    meta = COUNTY_METADATA.get(fips, {})
    icon = tier_icons.get(result.escalation_tier.value, '?')
    flags_str = ', '.join(result.flags.active_flags) if result.flags.active_flags else 'none'
    sparse_note = ' ⚠ sparse' if result.routed_to_sparse_queue else ''
    print(f"{meta.get('name','?'):<25} {result.css:<8.3f} {icon+' '+result.escalation_tier.value:<12} {result.confidence.percentage:<12.0%} {flags_str}{sparse_note}")

print()
print('⚠  No autonomous action taken. Human confirmation required before any intervention.')

ESCALATION SUMMARY
County                    CSS      Tier         Confidence   Flags
-----------------------------------------------------------------
Jefferson County          0.354    🟢 LOW        10%          bot_risk
Madison County            0.190    ⚪ NOISE      10%          media_context, bot_risk
Lowndes County            0.000    🟢 LOW        20%          rural_sparse, rural_bias ⚠ sparse

⚠  No autonomous action taken. Human confirmation required before any intervention.


## Step 6 — Audit Logging

Every evaluation generates an immutable audit record.
Four categories: what the system **saw**, what it **decided**, what the **human did**, and what **happened after**.

This is what makes the system accountable — if something goes wrong, every decision is reconstructable.

In [17]:
import tempfile, os

# Use temp file so notebook doesn't pollute the repo
tmp_log = tempfile.mktemp(suffix='.jsonl')
logger = AuditLogger(tmp_log)

audit_records = {}
for fips, result in css_results.items():
    record = logger.log_evaluation(result)
    audit_records[fips] = record

# Show one full audit record as example
example_fips = '01073'
example_record = audit_records[example_fips]
meta = COUNTY_METADATA[example_fips]

print(f'Sample audit record — {meta["name"]}:')
print('='*55)
record_dict = example_record.to_dict()
# Show key fields
display_fields = [
    'event_id', 'timestamp_utc', 'county_fips', 'n_posts',
    'sentiment_score', 'volume_score', 'volume_adjusted',
    'css', 'escalation_tier', 'active_flags',
    'confidence_pct', 'confidence_plain_language', 'confidence_visual_tier'
]
for field in display_fields:
    print(f'  {field:<35} {record_dict[field]}')

print()
print(f'Human review fields (populated when analyst reviews):')
print(f'  analyst_id:                          {record_dict["analyst_id"]}')
print(f'  analyst_action:                      {record_dict["analyst_action"]}')
print(f'  outcome_confirmed:                   {record_dict["outcome_confirmed"]}')

Sample audit record — Jefferson County:
  event_id                            31cb0667-a4dd-4795-a877-e33e7297389b
  timestamp_utc                       2026-03-04T18:05:24.878805Z
  county_fips                         01073
  n_posts                             40
  sentiment_score                     0.2845
  volume_score                        0.6458
  volume_adjusted                     0.6458
  css                                 0.3541
  escalation_tier                     LOW
  active_flags                        ['bot_risk']
  confidence_pct                      0.1
  confidence_plain_language           Signal driven by significant volume spike — Caution: flags active: bot_risk.
  confidence_visual_tier              GRAY

Human review fields (populated when analyst reviews):
  analyst_id:                          None
  analyst_action:                      None
  outcome_confirmed:                   None


## Step 7 — Simulating Human Review

The analyst reviews the Jefferson County signal and logs their decision.
This is the mandatory human-in-the-loop step — no score triggers action without this.

In [18]:
jefferson_record = audit_records['01073']
jefferson_result = css_results['01073']

print('ANALYST REVIEW INTERFACE')
print('='*55)
print(f'County:          Jefferson County (01073)')
print(f'CSS Score:       {jefferson_result.css:.3f}')
print(f'Tier:            {jefferson_result.escalation_tier.value}')
print(f'Confidence:      {jefferson_result.confidence.percentage:.0%}')
print(f'Active flags:    {jefferson_result.flags.active_flags or "none"}')
print(f'\nConfidence note: {jefferson_result.confidence.plain_language}')
print()

# Simulate analyst confirming the signal
success = logger.log_analyst_review(
    event_id=jefferson_record.event_id,
    analyst_id='analyst_dmw_001',
    action=AnalystAction.CONFIRMED,
    rationale=(
        'Reviewed de-identified post sample. Language patterns consistent with '
        'genuine community distress, not media-reactive. '
        'Sentiment driven by first-person expressions across 25+ unique accounts. '
        'Recommending outreach to county behavioral health coordinator.'
    )
)

print(f'Analyst review logged: {"✅ Success" if success else "❌ Failed"}')
print(f'Action:          CONFIRMED')
print(f'Audit ID:        {jefferson_record.event_id[:8]}...')
print()
print('⚠  This confirmation authorizes outreach — it does NOT automatically deploy resources.')
print('   Next step: County behavioral health coordinator notified for human decision on intervention.')

ANALYST REVIEW INTERFACE
County:          Jefferson County (01073)
CSS Score:       0.354
Tier:            LOW
Confidence:      10%
Active flags:    ['bot_risk']

Confidence note: Signal driven by significant volume spike — Caution: flags active: bot_risk.

Analyst review logged: ✅ Success
Action:          CONFIRMED
Audit ID:        31cb0667...

⚠  This confirmation authorizes outreach — it does NOT automatically deploy resources.
   Next step: County behavioral health coordinator notified for human decision on intervention.


## Step 8 — Governance Flags Deep Dive

Comparing how the same volume level is interpreted differently
when governance flags are present vs. absent.

In [19]:
from ai4mh.flags import FlagResult
from ai4mh.scoring import compute_volume_score

print('GOVERNANCE FLAG IMPACT DEMONSTRATION')
print('='*55)
print('Scenario: 60 posts in a county with baseline mean=20, std=8')
print()

baseline_mean, baseline_std, n_posts = 20, 8, 60
v_raw = compute_volume_score(n_posts, baseline_mean, baseline_std)

scenarios = [
    ('No flags', 1.0),
    ('Bot risk detected', 0.5),
    ('Media spike detected', 0.6),
    ('Both bot + media', 0.5),
]

print(f'{"Scenario":<30} {"Raw Volume":<14} {"Adjusted":<14} {"Reduction"}')
print('-'*65)
for scenario, discount in scenarios:
    adjusted = round(v_raw * discount, 3)
    reduction = f"{(1-discount)*100:.0f}%" if discount < 1.0 else 'none'
    print(f'{scenario:<30} {v_raw:<14.3f} {adjusted:<14.3f} {reduction}')

print()
print('Key: Volume is DISCOUNTED, not deleted.')
print('The human reviewer always sees the reason for the discount in plain language.')

GOVERNANCE FLAG IMPACT DEMONSTRATION
Scenario: 60 posts in a county with baseline mean=20, std=8

Scenario                       Raw Volume     Adjusted       Reduction
-----------------------------------------------------------------
No flags                       1.000          1.000          none
Bot risk detected              1.000          0.500          50%
Media spike detected           1.000          0.600          40%
Both bot + media               1.000          0.500          50%

Key: Volume is DISCOUNTED, not deleted.
The human reviewer always sees the reason for the discount in plain language.


## Step 9 — Rural vs Urban Threshold Comparison

Demonstrating how the tiered threshold system prevents rural communities
from being silenced by an urban-calibrated minimum sample size.

In [20]:
from ai4mh.flags import detect_rural_sparse

print('RURAL vs URBAN THRESHOLD DEMONSTRATION')
print('='*60)
print(f'Urban minimum posts: {config.MIN_POSTS_URBAN}')
print(f'Rural minimum posts: {config.MIN_POSTS_RURAL}')
print()

test_cases = [
    ('Jefferson Co (urban, 8 posts)',  '01073', 8,  2, 'Below urban threshold'),
    ('Jefferson Co (urban, 35 posts)', '01073', 35, 2, 'Above urban threshold'),
    ('Lowndes Co (rural, 8 posts)',    '01085', 8,  8, 'Below rural threshold'),
    ('Lowndes Co (rural, 20 posts)',   '01085', 20, 8, 'Above rural threshold'),
]

print(f'{"Scenario":<38} {"Flags":<30} {"Action"}')
print('-'*85)
for name, fips, n, rucc, expected in test_cases:
    result = detect_rural_sparse(fips, n, rucc)
    flags = ', '.join(result.active_flags) if result.active_flags else 'none'
    action = 'sparse queue' if 'rural_sparse' in result.active_flags else 'proceeds to scoring'
    print(f'{name:<38} {flags:<30} {action}')

print()
print('Rural counties with few posts → sparse queue (human review), NOT silence.')
print('Rural counties always carry rural_bias flag → model may underdetect distress.')

RURAL vs URBAN THRESHOLD DEMONSTRATION
Urban minimum posts: 30
Rural minimum posts: 15

Scenario                               Flags                          Action
-------------------------------------------------------------------------------------
Jefferson Co (urban, 8 posts)          rural_sparse                   sparse queue
Jefferson Co (urban, 35 posts)         none                           proceeds to scoring
Lowndes Co (rural, 8 posts)            rural_sparse, rural_bias       sparse queue
Lowndes Co (rural, 20 posts)           rural_bias                     proceeds to scoring

Rural counties with few posts → sparse queue (human review), NOT silence.
Rural counties always carry rural_bias flag → model may underdetect distress.


## Step 10 — Full Audit Log Review

All records written during this walkthrough, showing the complete provenance chain.

In [21]:
all_records = logger.get_all_records()

print(f'Total audit records written: {len(all_records)}')
print()

for i, rec in enumerate(all_records, 1):
    record_type = rec.get('record_type', 'evaluation')
    print(f'Record {i}: {record_type.upper()}')

    if record_type == 'evaluation':
        print(f"  County: {rec.get('county_fips')} | CSS: {rec.get('css')} | "
              f"Tier: {rec.get('escalation_tier')} | Flags: {rec.get('active_flags')}")
    elif record_type == 'analyst_review':
        print(f"  Event: {rec.get('event_id','')[:8]}... | "
              f"Action: {rec.get('analyst_action')} | "
              f"Analyst: {rec.get('analyst_id')}")
    print()

print('All records are append-only and tamper-evident.')
print('In production: stored in QLDB or equivalent, retained 7 years.')

# Cleanup temp file
os.remove(tmp_log)

Total audit records written: 4

Record 1: EVALUATION
  County: 01073 | CSS: 0.3541 | Tier: LOW | Flags: ['bot_risk']

Record 2: EVALUATION
  County: 01089 | CSS: 0.19 | Tier: NOISE | Flags: ['media_context', 'bot_risk']

Record 3: EVALUATION
  County: 01085 | CSS: 0.0 | Tier: LOW | Flags: ['rural_sparse', 'rural_bias']

Record 4: ANALYST_REVIEW
  Event: 31cb0667... | Action: CONFIRMED | Analyst: analyst_dmw_001

All records are append-only and tamper-evident.
In production: stored in QLDB or equivalent, retained 7 years.


## Summary

This walkthrough demonstrated the complete AI4MH governance pipeline:

| Step | Component | Key Design Decision |
|------|-----------|--------------------|
| 1-2 | Data loading | 72-hour rolling window, 3 county scenarios |
| 3 | Governance flags | Discount + flag, never delete |
| 4 | CSS scoring | Sentiment anchors; volume/geo modify confidence |
| 5 | Escalation | Tiered routing — no autonomous action |
| 6 | Audit logging | 4 categories: saw / decided / human did / outcome |
| 7 | Human review | Mandatory before any intervention |
| 8 | Flag impact | Volume discounted but visible to reviewer |
| 9 | Rural equity | Tiered thresholds — rural counties never silenced |
| 10 | Full audit trail | Every decision reconstructable |

---

**Roadmap for GSoC coding period:**
- Replace VADER with 2025 BERT classifier
- Reddit API (PRAW) live data integration
- Plotly Dash interactive dashboard with county heatmaps
- GeoPandas + Folium geospatial visualization
- Moran's I geographic clustering
- Quarterly equity audit reports

**GitHub:** https://github.com/Swasti2004/gsoc2026-ai4mh-governance